# Tutorial 6.2 – Fuzzy Linguistic Variables

This notebook continues from **Tutorial 6.1**.

In Tutorial 6.1, the raw EmotiBit CSV files were inspected, cleaned, synchronized, and saved as one clean sensor table. In this notebook, we use that cleaned table to create fuzzy linguistic variables and human-readable sensor interpretations.

The main idea is simple:

> Sensors produce numbers, but humans often think and communicate with words such as *low*, *normal*, *high*, *comfortable*, *active*, or *relaxed*.

This version does **not** require PostgreSQL, any database server, a username, or a password. DuckDB runs locally from Python and creates a small database file inside the notebook folder.


## Learning objectives

By the end of this tutorial, you should be able to:

1. load the clean sensor table created in Tutorial 6.1;
2. load the clean table into a local DuckDB database;
3. define fuzzy linguistic variables such as `low`, `normal`, `high`, `comfortable`, and `active`;
4. create a DuckDB view that adds fuzzy membership degrees to the raw data;
5. query the data using fuzzy linguistic terms;
6. produce a simple combined interpretation of the participant's physiological state;
7. run a basic sanity check for fuzzy membership values.


## Prerequisites

Before running this notebook:

1. Complete **Tutorial 6.1**. It should create:

```text
tutorial_data/processed/clean_emotibit_readings.csv
```

2. If you completed the linguistic self-perception questionnaire in **Tutorial 5.2**, its CSV file should be available in:

```text
tutorial_data/questionnaire/
```

The questionnaire is used near the end of this tutorial to compare a human linguistic self-report with the sensor-derived fuzzy linguistic interpretation.

**Install the required Python packages in the same environment where JupyterLab is running:**

```bash
pip install pandas numpy duckdb matplotlib jupyterlab nbconvert
```

This notebook uses Python 3, pandas, NumPy and DuckDB.

The fuzzy ranges used here are educational and dataset-specific. They are **not medical thresholds or diagnostic rules**.


## General workflow

The workflow of this notebook is:

```text
Clean sensor table from Tutorial 6.1
        ↓
Load into a local DuckDB database
        ↓
Create fuzzy membership functions
        ↓
Create fuzzy linguistic variables
        ↓
Run fuzzy SQL queries
        ↓
Create combined sensor interpretation
        ↓
Load the questionnaire from Tutorial 5.2
        ↓
Compare sensor-derived words with human self-perception
```

The final goal is not to claim that fuzzy logic reproduces human perception exactly. Instead, fuzzy linguistic variables provide a **semantic bridge** between precise numerical sensor measurements and the words people naturally use to describe their own experience.


In [2]:
# If needed, install the required packages in your virtual environment first.
# Run this in Terminal, not inside the notebook, if packages are missing:
# pip install pandas numpy duckdb matplotlib jupyterlab nbconvert

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

SHARED_DATA_DIR = Path("..") / "tutorial_data"
PROCESSED_DIR = SHARED_DATA_DIR / "processed"
QUESTIONNAIRE_DIR = SHARED_DATA_DIR / "questionnaire"
DATABASE_DIR = SHARED_DATA_DIR / "database"
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

clean_csv_path = PROCESSED_DIR / "clean_emotibit_readings.csv"

if not clean_csv_path.exists():
    raise FileNotFoundError(
        "The clean dataset was not found. Please run Tutorial 6.1 first.\n"
        f"Expected file: {clean_csv_path}"
    )

clean = pd.read_csv(clean_csv_path, parse_dates=["timestamp"])
print(f"Loaded clean sensor table from: {clean_csv_path}")
print(f"Rows: {len(clean)}")
clean.head()


Loaded clean sensor table from: ../tutorial_data/processed/clean_emotibit_readings.csv
Rows: 144


,timestamp,heart_rate,skin_temperature,eda,scr_frequency,ax,ay,az,accel_magnitude,movement_intensity
0,2026-04-26 20:34:21,71.196667,35.694333,0.030260,0.992350,0.298933,-0.375956,0.886178,1.007977,0.007977
1,2026-04-26 20:34:22,71.196667,35.679500,0.030260,0.963933,0.299360,-0.375320,0.885320,1.007267,0.007550
2,2026-04-26 20:34:23,71.196667,35.664000,0.030260,0.930867,0.300480,-0.372920,0.886240,1.007533,0.009006
3,2026-04-26 20:34:24,71.196667,35.702625,0.030259,0.898933,0.299520,-0.377680,0.885200,1.007984,0.008095
4,2026-04-26 20:34:25,102.560000,35.716286,0.030259,0.868100,0.312538,-0.371000,0.881346,1.006098,0.007859


## Part C — Load the clean table into DuckDB

This section uses **DuckDB** as the database layer. DuckDB works directly inside Python and is well suited for CSV/data-analysis workflows.

The notebook creates a local database file in the shared course-data folder:

```text
tutorial_data/database/tutorial_6_2_fuzzy.duckdb
```

Inside that database, we create a table named:

```text
raw_emotibit_readings
```

The database file and the questionnaire CSV remain separate files. Tutorial 6.2 simply reads both when they are needed.


In [3]:
# DuckDB database path inside the shared course-data folder.
DB_PATH = DATABASE_DIR / "tutorial_6_2_fuzzy.duckdb"
print(f"DuckDB database file will be created/used here: {DB_PATH}")


DuckDB database file will be created/used here: ../tutorial_data/database/tutorial_6_2_fuzzy.duckdb


In [4]:
# Create/connect to DuckDB and load the clean DataFrame.

import duckdb

duckdb_ready = False
conn = None

try:
    conn = duckdb.connect(str(DB_PATH))
    conn.register("clean_df", clean)
    conn.execute("CREATE OR REPLACE TABLE raw_emotibit_readings AS SELECT * FROM clean_df")
    row_count = conn.execute("SELECT COUNT(*) FROM raw_emotibit_readings").fetchone()[0]
    duckdb_ready = True
    print(f"Loaded {row_count} rows into DuckDB table: raw_emotibit_readings")
except Exception as exc:
    print("DuckDB connection/loading did not work yet.")
    print("Reason:", exc)
    print("Check that DuckDB is installed in the same environment as JupyterLab.")
    duckdb_ready = False


Loaded 144 rows into DuckDB table: raw_emotibit_readings


## Part D — Create fuzzy membership functions and linguistic variables

A fuzzy membership function converts a numeric value into a degree between `0` and `1`.

For example, a heart rate value can be partly `normal` and partly `high` at the same time. This is the key difference from a crisp threshold.

We first create a triangular membership function in DuckDB using a SQL macro.


In [5]:
create_fuzzy_function_sql = """
CREATE OR REPLACE MACRO fuzzy_tri_membership(x, a, b, c) AS
    CASE
        WHEN x IS NULL OR a >= b OR b >= c THEN NULL
        WHEN x <= a OR x >= c THEN 0.0
        WHEN x = b THEN 1.0
        WHEN x > a AND x < b THEN (x - a) / (b - a)
        WHEN x > b AND x < c THEN (c - x) / (c - b)
        ELSE 0.0
    END;
"""

if duckdb_ready:
    conn.execute(create_fuzzy_function_sql)
    print("Created DuckDB SQL macro: fuzzy_tri_membership")
else:
    print("DuckDB is not ready yet. The SQL macro is stored in create_fuzzy_function_sql.")


Created DuckDB SQL macro: fuzzy_tri_membership


### Linguistic variables used in this notebook

For the final comparison with the Tutorial 5.2 questionnaire, each concept uses **five linguistic terms**.

The ranges are educational and dataset-specific. They are **not medical thresholds**.

| Sensor value | Five linguistic terms |
|---|---|
| `heart_rate` | `very low`, `low`, `normal`, `high`, `very high` |
| `skin_temperature` | `cool`, `slightly cool`, `comfortable`, `slightly warm`, `warm` |
| `scr_frequency` | `very calm`, `calm`, `moderate`, `elevated`, `highly elevated` |
| `movement_intensity` | `still`, `slight`, `moderate`, `active`, `very active` |

A value can belong partly to two neighboring terms at the same time. This overlapping membership is what makes the classification fuzzy rather than a rigid threshold classification.

The next cell creates a DuckDB view where each sensor row receives fuzzy membership columns.


In [6]:
create_fuzzy_view_sql = """
CREATE OR REPLACE VIEW fuzzy_emotibit_readings AS
SELECT
    timestamp,
    heart_rate,
    skin_temperature,
    eda,
    scr_frequency,
    ax,
    ay,
    az,
    accel_magnitude,
    movement_intensity,

    -- Heart-rate memberships: five overlapping linguistic terms
    fuzzy_tri_membership(heart_rate, 30, 42, 58) AS hr_very_low,
    fuzzy_tri_membership(heart_rate, 42, 58, 74) AS hr_low,
    fuzzy_tri_membership(heart_rate, 58, 78, 100) AS hr_normal,
    fuzzy_tri_membership(heart_rate, 86, 108, 130) AS hr_high,
    fuzzy_tri_membership(heart_rate, 112, 138, 170) AS hr_very_high,

    -- Skin/contact-temperature memberships
    fuzzy_tri_membership(skin_temperature, 33.8, 34.8, 35.5) AS temp_cool,
    fuzzy_tri_membership(skin_temperature, 34.8, 35.4, 35.9) AS temp_slightly_cool,
    fuzzy_tri_membership(skin_temperature, 35.4, 35.9, 36.3) AS temp_comfortable,
    fuzzy_tri_membership(skin_temperature, 35.9, 36.3, 36.7) AS temp_slightly_warm,
    fuzzy_tri_membership(skin_temperature, 36.3, 36.8, 37.4) AS temp_warm,

    -- Skin-conductance-response frequency / arousal-related memberships
    fuzzy_tri_membership(scr_frequency, 0.00, 0.02, 0.10) AS arousal_very_calm,
    fuzzy_tri_membership(scr_frequency, 0.03, 0.10, 0.22) AS arousal_calm,
    fuzzy_tri_membership(scr_frequency, 0.12, 0.28, 0.48) AS arousal_moderate,
    fuzzy_tri_membership(scr_frequency, 0.32, 0.58, 0.85) AS arousal_elevated,
    fuzzy_tri_membership(scr_frequency, 0.65, 0.95, 1.30) AS arousal_highly_elevated,

    -- Movement memberships
    fuzzy_tri_membership(movement_intensity, 0.00, 0.02, 0.07) AS movement_still,
    fuzzy_tri_membership(movement_intensity, 0.03, 0.09, 0.20) AS movement_slight,
    fuzzy_tri_membership(movement_intensity, 0.10, 0.28, 0.50) AS movement_moderate,
    fuzzy_tri_membership(movement_intensity, 0.32, 0.62, 0.95) AS movement_active,
    fuzzy_tri_membership(movement_intensity, 0.75, 1.10, 1.50) AS movement_very_active,

    -- Backward-compatible broad aliases used by the existing tutorial queries
    fuzzy_tri_membership(movement_intensity, 0.04, 0.20, 0.50) AS movement_moving
FROM raw_emotibit_readings;
"""

if duckdb_ready:
    conn.execute(create_fuzzy_view_sql)
    print("Created DuckDB view: fuzzy_emotibit_readings")
else:
    print("DuckDB is not ready yet. The SQL view is stored in create_fuzzy_view_sql.")


Created DuckDB view: fuzzy_emotibit_readings


In [7]:
# Preview the fuzzy view.
if duckdb_ready:
    preview = conn.execute("SELECT * FROM fuzzy_emotibit_readings LIMIT 2").df()
    display(preview)
else:
    print("Run the DuckDB connection cells first, then rerun this preview cell.")


,timestamp,heart_rate,skin_temperature,eda,scr_frequency,ax,ay,az,accel_magnitude,movement_intensity,hr_very_low,hr_low,hr_normal,hr_high,hr_very_high,temp_cool,temp_slightly_cool,temp_comfortable,temp_slightly_warm,temp_warm,arousal_very_calm,arousal_calm,arousal_moderate,arousal_elevated,arousal_highly_elevated,movement_still,movement_slight,movement_moderate,movement_active,movement_very_active,movement_moving
0,2026-04-26 20:34:21,71.196667,35.694333,0.03026,0.992350,0.298933,-0.375956,0.886178,1.007977,0.007977,0.0,0.175208,0.659833,0.0,0.0,0.0,0.411333,0.588667,0.0,0.0,0.0,0.0,0.0,0.0,0.87900,0.398873,0.0,0.0,0.0,0.0,0.0
1,2026-04-26 20:34:22,71.196667,35.679500,0.03026,0.963933,0.299360,-0.375320,0.885320,1.007267,0.007550,0.0,0.175208,0.659833,0.0,0.0,0.0,0.441000,0.559000,0.0,0.0,0.0,0.0,0.0,0.0,0.96019,0.377512,0.0,0.0,0.0,0.0,0.0


## Part E — Query the data using fuzzy linguistic terms

Now DuckDB can answer questions using fuzzy columns.

Instead of asking only crisp questions such as:

```sql
WHERE heart_rate > 100
```

we can ask fuzzy linguistic questions such as:

```sql
WHERE hr_high > 0.6
```

This means: show the moments where the heart rate belongs strongly to the linguistic term `high`.


In [8]:
def run_sql(query: str) -> pd.DataFrame:
    # Run a SQL query in DuckDB and return a pandas DataFrame.
    if not duckdb_ready:
        print("DuckDB is not ready. Rerun the DuckDB connection/loading cells first.")
        return pd.DataFrame()
    return conn.execute(query).df()


### Query 1 — Moments where heart rate is high


In [9]:
query_high_hr = """
SELECT
    timestamp,
    ROUND(heart_rate, 2) AS heart_rate,
    ROUND(hr_high, 3) AS hr_high,
    ROUND(hr_normal, 3) AS hr_normal
FROM fuzzy_emotibit_readings
WHERE hr_high > 0.90
ORDER BY hr_high DESC, timestamp
LIMIT 3;
"""
run_sql(query_high_hr)


,timestamp,heart_rate,hr_high,hr_normal
0,2026-04-26 20:35:43,107.85,0.993,0.0
1,2026-04-26 20:35:48,108.23,0.990,0.0
2,2026-04-26 20:35:41,108.34,0.985,0.0


### Query 2 — Moments where movement is active


In [10]:
query_active_movement = """
SELECT
    timestamp,
    ROUND(movement_intensity, 3) AS movement_intensity,
    ROUND(movement_active, 3) AS movement_active,
    ROUND(movement_moving, 3) AS movement_moving,
    ROUND(movement_still, 3) AS movement_still
FROM fuzzy_emotibit_readings
WHERE movement_active > 0.80
ORDER BY movement_active DESC, timestamp
LIMIT 3;
"""
run_sql(query_active_movement)


,timestamp,movement_intensity,movement_active,movement_moving,movement_still
0,2026-04-26 20:35:00,0.616,0.988,0.0,0.0
1,2026-04-26 20:35:20,0.614,0.979,0.0,0.0
2,2026-04-26 20:35:30,0.630,0.971,0.0,0.0


### Query 3 — Moments where the arousal-related signal is elevated

Here we use `scr_frequency` as the arousal-related signal. This is only a tutorial interpretation and should not be treated as a medical or psychological diagnosis.


In [11]:
query_elevated_arousal = """
SELECT
    timestamp,
    ROUND(scr_frequency, 3) AS scr_frequency,
    ROUND(arousal_elevated, 3) AS arousal_elevated,
    ROUND(arousal_moderate, 3) AS arousal_moderate,
    ROUND(arousal_calm, 3) AS arousal_calm
FROM fuzzy_emotibit_readings
WHERE arousal_elevated > 0.60
ORDER BY arousal_elevated DESC, timestamp
LIMIT 3;
"""
run_sql(query_elevated_arousal)


,timestamp,scr_frequency,arousal_elevated,arousal_moderate,arousal_calm
0,2026-04-26 20:34:37,0.578,0.992,0.0,0.0
1,2026-04-26 20:34:36,0.598,0.933,0.0,0.0
2,2026-04-26 20:34:38,0.558,0.914,0.0,0.0


### Query 4 — Elevated arousal signal while movement remains low

This query combines two linguistic ideas:

- the arousal-related signal is elevated;
- movement is still or low.

In plain words, this means: **there is a stronger arousal signal, but the participant is not moving much**. This may help separate physical activity from a possible internal activation pattern. The result should still be interpreted carefully.


In [12]:
query_elevated_arousal_low_movement = """
SELECT
    timestamp,
    ROUND(scr_frequency, 3) AS scr_frequency,
    ROUND(movement_intensity, 3) AS movement_intensity,
    ROUND(arousal_elevated, 3) AS arousal_elevated,
    ROUND(movement_still, 3) AS movement_still,
    ROUND(LEAST(arousal_elevated, movement_still), 3) AS elevated_arousal_low_movement_score
FROM fuzzy_emotibit_readings
WHERE LEAST(arousal_elevated, movement_still) > 0.55
ORDER BY elevated_arousal_low_movement_score DESC, timestamp
LIMIT 5;
"""
run_sql(query_elevated_arousal_low_movement)


,timestamp,scr_frequency,movement_intensity,arousal_elevated,movement_still,elevated_arousal_low_movement_score
0,2026-04-26 20:34:37,0.578,0.025,0.992,0.897,0.897
1,2026-04-26 20:34:35,0.612,0.015,0.880,0.772,0.772
2,2026-04-26 20:34:34,0.634,0.014,0.800,0.700,0.700
3,2026-04-26 20:34:41,0.497,0.023,0.679,0.935,0.679
4,2026-04-26 20:34:38,0.558,0.013,0.914,0.647,0.647


### Query 5 — Physical activity pattern

This query combines higher movement with higher heart rate.

In plain words, this means: **the participant appears to be moving more, and heart rate is also higher**.


In [13]:
query_physical_activity = """
SELECT
    timestamp,
    ROUND(heart_rate, 2) AS heart_rate,
    ROUND(movement_intensity, 3) AS movement_intensity,
    ROUND(hr_high, 3) AS hr_high,
    ROUND(movement_active, 3) AS movement_active,
    ROUND(LEAST(hr_high, movement_active), 3) AS physically_active_score
FROM fuzzy_emotibit_readings
WHERE LEAST(hr_high, movement_active) > 0.25
ORDER BY physically_active_score DESC, timestamp
LIMIT 5;
"""
run_sql(query_physical_activity)


,timestamp,heart_rate,movement_intensity,hr_high,movement_active,physically_active_score
0,2026-04-26 20:34:53,110.69,0.691,0.878,0.786,0.786
1,2026-04-26 20:34:51,102.78,0.565,0.763,0.816,0.763


### Query 6 — Count seconds by strongest heart-rate term

This query converts fuzzy memberships into a simple dominant label for quick summary. The underlying membership degrees are still available in the view.


In [14]:
query_hr_summary = """
WITH labelled AS (
    SELECT
        CASE
            WHEN hr_very_low >= hr_low AND hr_very_low >= hr_normal
             AND hr_very_low >= hr_high AND hr_very_low >= hr_very_high THEN 'very low'
            WHEN hr_low >= hr_very_low AND hr_low >= hr_normal
             AND hr_low >= hr_high AND hr_low >= hr_very_high THEN 'low'
            WHEN hr_normal >= hr_very_low AND hr_normal >= hr_low
             AND hr_normal >= hr_high AND hr_normal >= hr_very_high THEN 'normal'
            WHEN hr_high >= hr_very_low AND hr_high >= hr_low
             AND hr_high >= hr_normal AND hr_high >= hr_very_high THEN 'high'
            ELSE 'very high'
        END AS strongest_hr_label
    FROM fuzzy_emotibit_readings
)
SELECT strongest_hr_label, COUNT(*) AS seconds
FROM labelled
GROUP BY strongest_hr_label
ORDER BY seconds DESC;
"""
run_sql(query_hr_summary)


,strongest_hr_label,seconds
0,normal,64
1,low,50
2,high,19
3,very low,10
4,very high,1


## Part F — Combined interpretation

The individual fuzzy values are useful, but Computing with Words becomes more meaningful when we combine them into a higher-level interpretation.

Here we define four simple interpretation scores:

| Interpretation label | Simple fuzzy rule | Plain meaning |
|---|---|---|
| `relaxed_pattern` | heart rate normal + arousal calm + movement still | heart rate is normal, arousal signal is calm, and movement is low |
| `engaged_pattern` | heart rate normal/high + arousal moderate + movement low/moderate | body signals show moderate engagement without strong movement |
| `physical_activity_pattern` | heart rate high + movement active | higher heart rate appears together with more movement |
| `elevated_arousal_low_movement` | arousal elevated + movement still | arousal signal is higher, but movement remains low / physically still |

We use `LEAST(...)` as a fuzzy AND and `GREATEST(...)` as a fuzzy OR.

These interpretations are **not diagnoses**. They are tutorial labels that make the fuzzy sensor pattern easier to read.


In [15]:
create_interpretation_view_sql = """
CREATE OR REPLACE VIEW fuzzy_emotibit_interpretation AS
WITH scores AS (
    SELECT
        *,
        LEAST(hr_normal, arousal_calm, movement_still) AS relaxed_score,
        LEAST(GREATEST(hr_normal, hr_high), arousal_moderate, GREATEST(movement_still, movement_moving)) AS engaged_score,
        LEAST(hr_high, movement_active) AS physically_active_score,
        LEAST(arousal_elevated, movement_still) AS elevated_arousal_low_movement_score
    FROM fuzzy_emotibit_readings
), labelled AS (
    SELECT
        *,
        CASE
            WHEN relaxed_score >= engaged_score
             AND relaxed_score >= physically_active_score
             AND relaxed_score >= elevated_arousal_low_movement_score THEN 'relaxed_pattern'
            WHEN engaged_score >= relaxed_score
             AND engaged_score >= physically_active_score
             AND engaged_score >= elevated_arousal_low_movement_score THEN 'engaged_pattern'
            WHEN physically_active_score >= relaxed_score
             AND physically_active_score >= engaged_score
             AND physically_active_score >= elevated_arousal_low_movement_score THEN 'physical_activity_pattern'
            ELSE 'elevated_arousal_low_movement'
        END AS strongest_interpretation_label
    FROM scores
)
SELECT
    timestamp,
    heart_rate,
    skin_temperature,
    scr_frequency,
    movement_intensity,
    relaxed_score,
    engaged_score,
    physically_active_score,
    elevated_arousal_low_movement_score,
    strongest_interpretation_label,
    CASE strongest_interpretation_label
        WHEN 'relaxed_pattern' THEN 'Relaxed pattern: heart rate is mostly normal, arousal signal is calm, and movement is low.'
        WHEN 'engaged_pattern' THEN 'Engaged pattern: the body signals show moderate engagement without strong movement.'
        WHEN 'physical_activity_pattern' THEN 'Physical activity pattern: higher heart rate appears together with more movement.'
        ELSE 'Elevated arousal with low movement: the arousal-related signal is higher, but movement remains low / physically still.'
    END AS plain_language_interpretation
FROM labelled;
"""

if duckdb_ready:
    conn.execute(create_interpretation_view_sql)
    print("Created DuckDB view: fuzzy_emotibit_interpretation")
else:
    print("DuckDB is not ready yet. The interpretation SQL is stored in create_interpretation_view_sql.")


Created DuckDB view: fuzzy_emotibit_interpretation


In [16]:
query_interpretation_preview = """
SELECT
    timestamp,
    ROUND(heart_rate, 2) AS heart_rate,
    ROUND(skin_temperature, 2) AS skin_temperature,
    ROUND(scr_frequency, 3) AS scr_frequency,
    ROUND(movement_intensity, 3) AS movement_intensity,
    ROUND(relaxed_score, 3) AS relaxed_score,
    ROUND(engaged_score, 3) AS engaged_score,
    ROUND(physically_active_score, 3) AS physically_active_score,
    ROUND(elevated_arousal_low_movement_score, 3) AS elevated_arousal_low_movement_score,
    strongest_interpretation_label,
    plain_language_interpretation
FROM fuzzy_emotibit_interpretation
ORDER BY timestamp
LIMIT 3;
"""
run_sql(query_interpretation_preview)


,timestamp,heart_rate,skin_temperature,scr_frequency,movement_intensity,relaxed_score,engaged_score,physically_active_score,elevated_arousal_low_movement_score,strongest_interpretation_label,plain_language_interpretation
0,2026-04-26 20:34:21,71.2,35.69,0.992,0.008,0.0,0.0,0.0,0.0,relaxed_pattern,"Relaxed pattern: heart rate is mostly normal, ..."
1,2026-04-26 20:34:22,71.2,35.68,0.964,0.008,0.0,0.0,0.0,0.0,relaxed_pattern,"Relaxed pattern: heart rate is mostly normal, ..."
2,2026-04-26 20:34:23,71.2,35.66,0.931,0.009,0.0,0.0,0.0,0.0,relaxed_pattern,"Relaxed pattern: heart rate is mostly normal, ..."


In [17]:
query_interpretation_summary = """
SELECT strongest_interpretation_label, plain_language_interpretation, COUNT(*) AS seconds
FROM fuzzy_emotibit_interpretation
GROUP BY strongest_interpretation_label, plain_language_interpretation
ORDER BY seconds DESC;
"""
run_sql(query_interpretation_summary)


,strongest_interpretation_label,plain_language_interpretation,seconds
0,relaxed_pattern,"Relaxed pattern: heart rate is mostly normal, ...",114
1,elevated_arousal_low_movement,Elevated arousal with low movement: the arousa...,18
2,physical_activity_pattern,Physical activity pattern: higher heart rate a...,7
3,engaged_pattern,Engaged pattern: the body signals show moderat...,5


### Human-readable sentence for one selected timestamp

The next cell takes one row and writes a short explanation in normal language.


In [18]:
def describe_interpretation(row: pd.Series) -> str:
    return (
        f"At {row['timestamp']}, the participant's heart rate was approximately "
        f"{row['heart_rate']:.1f} bpm, skin temperature was {row['skin_temperature']:.2f} °C, "
        f"SCR frequency was {row['scr_frequency']:.3f}, and movement intensity was {row['movement_intensity']:.3f}. "
        f"The strongest combined fuzzy interpretation is: {row['plain_language_interpretation']}"
    )

if duckdb_ready:
    one_row = conn.execute("SELECT * FROM fuzzy_emotibit_interpretation ORDER BY timestamp LIMIT 1").df().iloc[0]
    print(describe_interpretation(one_row))
else:
    print("DuckDB is not ready yet. Run the DuckDB cells first.")


At 2026-04-26 20:34:21, the participant's heart rate was approximately 71.2 bpm, skin temperature was 35.69 °C, SCR frequency was 0.992, and movement intensity was 0.008. The strongest combined fuzzy interpretation is: Relaxed pattern: heart rate is mostly normal, arousal signal is calm, and movement is low.


## Part G — Compare Fuzzy Sensor Language with Human Self-Perception

Tutorial 5.2 asked you to describe the recording session using linguistic choices **before viewing the processed sensor plots**.

This section brings the two descriptions together:

```text
Sensor measurements
        ↓
Fuzzy linguistic interpretation
        +
Human questionnaire response
        ↓
Linguistic comparison
```

The comparison does **not** convert the questionnaire into hidden numbers. Both sides remain linguistic.

Agreement is also not treated as a test of whether one source is “correct”. Physiological signals and self-perception can agree, partially agree, or differ. Each outcome can be informative.


### Create one linguistic summary for the recording session

The sensor view contains one fuzzy interpretation per time step. For comparison with one questionnaire response, we first average the membership degrees over the recording session and select the strongest linguistic term for each concept.

The full membership values remain available, so the fuzzy overlap is not lost.


In [19]:
if duckdb_ready:
    session_membership_query = """
    SELECT
        AVG(hr_very_low) AS hr_very_low,
        AVG(hr_low) AS hr_low,
        AVG(hr_normal) AS hr_normal,
        AVG(hr_high) AS hr_high,
        AVG(hr_very_high) AS hr_very_high,

        AVG(temp_cool) AS temp_cool,
        AVG(temp_slightly_cool) AS temp_slightly_cool,
        AVG(temp_comfortable) AS temp_comfortable,
        AVG(temp_slightly_warm) AS temp_slightly_warm,
        AVG(temp_warm) AS temp_warm,

        AVG(movement_still) AS movement_still,
        AVG(movement_slight) AS movement_slight,
        AVG(movement_moderate) AS movement_moderate,
        AVG(movement_active) AS movement_active,
        AVG(movement_very_active) AS movement_very_active,

        AVG(arousal_very_calm) AS arousal_very_calm,
        AVG(arousal_calm) AS arousal_calm,
        AVG(arousal_moderate) AS arousal_moderate,
        AVG(arousal_elevated) AS arousal_elevated,
        AVG(arousal_highly_elevated) AS arousal_highly_elevated
    FROM fuzzy_emotibit_readings;
    """

    session_membership = conn.execute(session_membership_query).df().iloc[0]

    groups = {
        "heartbeat": {
            "very low": session_membership["hr_very_low"],
            "low": session_membership["hr_low"],
            "normal": session_membership["hr_normal"],
            "high": session_membership["hr_high"],
            "very high": session_membership["hr_very_high"],
        },
        "skin_temperature": {
            "cool": session_membership["temp_cool"],
            "slightly cool": session_membership["temp_slightly_cool"],
            "comfortable": session_membership["temp_comfortable"],
            "slightly warm": session_membership["temp_slightly_warm"],
            "warm": session_membership["temp_warm"],
        },
        "movement": {
            "still": session_membership["movement_still"],
            "slight": session_membership["movement_slight"],
            "moderate": session_membership["movement_moderate"],
            "active": session_membership["movement_active"],
            "very active": session_membership["movement_very_active"],
        },
        "activation_arousal": {
            "very calm": session_membership["arousal_very_calm"],
            "calm": session_membership["arousal_calm"],
            "moderate": session_membership["arousal_moderate"],
            "elevated": session_membership["arousal_elevated"],
            "highly elevated": session_membership["arousal_highly_elevated"],
        },
    }

    fuzzy_session_labels = {
        concept: max(term_scores, key=term_scores.get)
        for concept, term_scores in groups.items()
    }

    fuzzy_session_summary = pd.DataFrame([
        {
            "concept": concept,
            "strongest_fuzzy_term": fuzzy_session_labels[concept],
            "membership": round(groups[concept][fuzzy_session_labels[concept]], 3)
        }
        for concept in groups
    ])

    display(fuzzy_session_summary)
else:
    print("DuckDB is not ready yet. Run the DuckDB and fuzzy-view cells first.")


,concept,strongest_fuzzy_term,membership
0,heartbeat,normal,0.368
1,skin_temperature,comfortable,0.676
2,movement,still,0.346
3,activation_arousal,very calm,0.334


### Load the questionnaire saved in Tutorial 5.2

Tutorial 5.2 saves questionnaire responses in:

```text
tutorial_data/questionnaire/
```

If several questionnaire CSV files exist, the next cell automatically selects the **most recently modified** one and shows which file was loaded.

No manual copying between tutorial folders is required when the tutorials are used inside the common course workspace.


In [20]:
questionnaire_files = sorted(
    QUESTIONNAIRE_DIR.glob("questionnaire_*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
) if QUESTIONNAIRE_DIR.exists() else []

questionnaire = None
questionnaire_path = None

if questionnaire_files:
    questionnaire_path = questionnaire_files[0]
    questionnaire = pd.read_csv(questionnaire_path)
    print(f"Loaded questionnaire: {questionnaire_path}")
    display(questionnaire)
else:
    print("No Tutorial 5.2 questionnaire file was found yet.")
    print(f"Expected folder: {QUESTIONNAIRE_DIR}")
    print("Complete and save the questionnaire in Tutorial 5.2, then run this cell again.")


Loaded questionnaire: ../tutorial_data/questionnaire/questionnaire_1.csv


,session_id,questionnaire_time,heartbeat_perception,skin_temperature_perception,movement_perception,activation_arousal_perception
0,1,2026-08-26T08:15:10,normal / usual,comfortable / neutral,slightly moving,neutral / moderately alert


### Compare the two linguistic descriptions

The questionnaire wording and the sensor-side fuzzy wording are not identical in every case. The next cell therefore maps only **equivalent linguistic phrases** to a common wording.

For example:

- `slightly faster than usual` → `high`
- `comfortable / neutral` → `comfortable`
- `moderately moving` → `moderate`
- `activated / alert` → `elevated`

This is a semantic mapping between words, not a hidden 1–5 score.

The comparison uses three plain-language outcomes:

- **agreement** — same linguistic region;
- **close / neighboring description** — adjacent semantic descriptions;
- **different description** — a clearer difference between sensor interpretation and self-report.


In [21]:
if questionnaire is not None and duckdb_ready:
    q = questionnaire.iloc[-1]

    questionnaire_to_common = {
        "heartbeat": {
            "much slower than usual": "very low",
            "slightly slower than usual": "low",
            "normal / usual": "normal",
            "slightly faster than usual": "high",
            "much faster than usual": "very high",
        },
        "skin_temperature": {
            "cool": "cool",
            "slightly cool": "slightly cool",
            "comfortable / neutral": "comfortable",
            "slightly warm": "slightly warm",
            "warm": "warm",
        },
        "movement": {
            "completely still": "still",
            "slightly moving": "slight",
            "moderately moving": "moderate",
            "actively moving": "active",
            "very active": "very active",
        },
        "activation_arousal": {
            "very calm": "very calm",
            "calm": "calm",
            "neutral / moderately alert": "moderate",
            "activated / alert": "elevated",
            "very activated / alert": "highly elevated",
        },
    }

    questionnaire_columns = {
        "heartbeat": "heartbeat_perception",
        "skin_temperature": "skin_temperature_perception",
        "movement": "movement_perception",
        "activation_arousal": "activation_arousal_perception",
    }

    neighboring_pairs = {
        "heartbeat": {
            frozenset(("very low", "low")),
            frozenset(("low", "normal")),
            frozenset(("normal", "high")),
            frozenset(("high", "very high")),
        },
        "skin_temperature": {
            frozenset(("cool", "slightly cool")),
            frozenset(("slightly cool", "comfortable")),
            frozenset(("comfortable", "slightly warm")),
            frozenset(("slightly warm", "warm")),
        },
        "movement": {
            frozenset(("still", "slight")),
            frozenset(("slight", "moderate")),
            frozenset(("moderate", "active")),
            frozenset(("active", "very active")),
        },
        "activation_arousal": {
            frozenset(("very calm", "calm")),
            frozenset(("calm", "moderate")),
            frozenset(("moderate", "elevated")),
            frozenset(("elevated", "highly elevated")),
        },
    }

    concept_names = {
        "heartbeat": "Heartbeat",
        "skin_temperature": "Skin temperature",
        "movement": "Movement",
        "activation_arousal": "Activation / arousal",
    }

    comparison_rows = []

    for concept in ["heartbeat", "skin_temperature", "movement", "activation_arousal"]:
        human_original = str(q[questionnaire_columns[concept]]).strip()
        human_common = questionnaire_to_common[concept].get(human_original, human_original)
        fuzzy_term = fuzzy_session_labels[concept]

        if fuzzy_term == human_common:
            relation = "agreement"
        elif frozenset((fuzzy_term, human_common)) in neighboring_pairs[concept]:
            relation = "close / neighboring description"
        else:
            relation = "different description"

        comparison_rows.append({
            "concept": concept_names[concept],
            "sensor_fuzzy_interpretation": fuzzy_term,
            "human_self_report": human_original,
            "comparison": relation,
        })

    comparison_table = pd.DataFrame(comparison_rows)
    display(comparison_table)
else:
    print("The comparison needs both the fuzzy sensor summary and a Tutorial 5.2 questionnaire file.")


,concept,sensor_fuzzy_interpretation,human_self_report,comparison
0,Heartbeat,normal,normal / usual,agreement
1,Skin temperature,comfortable,comfortable / neutral,agreement
2,Movement,still,slightly moving,close / neighboring description
3,Activation / arousal,very calm,neutral / moderately alert,different description


### Human-readable comparison

The final cell turns the comparison table into a short linguistic interpretation.

The purpose is to make the relationship between sensor measurements and human self-perception easier to discuss. A difference is not automatically an error: the physiological signal and the person's own perception are two different observations of the same recording session.


In [22]:
if questionnaire is not None and duckdb_ready:
    sentence_parts = []

    for _, row in comparison_table.iterrows():
        sentence_parts.append(
            f"{row['concept']}: the sensor-derived fuzzy description is "
            f"'{row['sensor_fuzzy_interpretation']}', while the self-report is "
            f"'{row['human_self_report']}' — {row['comparison']}."
        )

    print("Sensor–self-perception comparison\n")
    for sentence in sentence_parts:
        print("•", sentence)

    exact_count = (comparison_table["comparison"] == "agreement").sum()
    close_count = (comparison_table["comparison"] == "close / neighboring description").sum()

    print("\nOverall interpretation:")
    if exact_count + close_count >= 3:
        print(
            "The sensor-derived fuzzy language is broadly close to the participant's "
            "linguistic self-report, while still preserving small differences between "
            "physiological measurement and personal perception."
        )
    elif exact_count + close_count >= 2:
        print(
            "The sensor-derived fuzzy language and the self-report show partial agreement. "
            "Some aspects are close, while other physiological signals differ from the "
            "participant's own perception."
        )
    else:
        print(
            "The sensor-derived fuzzy language differs noticeably from the self-report in "
            "this session. This does not mean that either source is wrong; it highlights "
            "a gap between physiological measurements and perceived experience."
        )


Sensor–self-perception comparison

• Heartbeat: the sensor-derived fuzzy description is 'normal', while the self-report is 'normal / usual' — agreement.
• Skin temperature: the sensor-derived fuzzy description is 'comfortable', while the self-report is 'comfortable / neutral' — agreement.
• Movement: the sensor-derived fuzzy description is 'still', while the self-report is 'slightly moving' — close / neighboring description.
• Activation / arousal: the sensor-derived fuzzy description is 'very calm', while the self-report is 'neutral / moderately alert' — different description.

Overall interpretation:
The sensor-derived fuzzy language is broadly close to the participant's linguistic self-report, while still preserving small differences between physiological measurement and personal perception.


### Why fuzzy language helps here

A crisp rule could force a measurement into one category immediately at a fixed boundary. For example, two almost identical measurements on opposite sides of a threshold could receive completely different labels.

Fuzzy membership allows neighboring linguistic terms to overlap. A measurement can therefore be partly `normal` and partly `high`, or partly `comfortable` and partly `slightly warm`.

This does **not** make the fuzzy interpretation identical to human perception. It makes the numerical sensor description more gradual and more naturally comparable with the linguistic terms used in the questionnaire.


## Part G — Basic sanity check

For this first version, we keep evaluation simple. We only check that fuzzy membership values stay inside the valid range `[0, 1]`.

A larger evaluation could later test more properties, but this basic check is enough for the first tutorial draft.


In [23]:
san_check_sql = """
SELECT
    MIN(v) AS minimum_membership,
    MAX(v) AS maximum_membership
FROM (
    SELECT unnest([
        hr_low, hr_normal, hr_high,
        temp_cool, temp_comfortable, temp_warm,
        arousal_calm, arousal_moderate, arousal_elevated,
        movement_still, movement_moving, movement_active
    ]) AS v
    FROM fuzzy_emotibit_readings
) memberships;
"""

if duckdb_ready:
    sanity = run_sql(san_check_sql)
    display(sanity)
    min_v = sanity.loc[0, "minimum_membership"]
    max_v = sanity.loc[0, "maximum_membership"]
    if min_v >= 0 and max_v <= 1:
        print("Sanity check passed: all fuzzy membership values are between 0 and 1.")
    else:
        print("Sanity check failed: at least one membership value is outside [0, 1].")
else:
    print("DuckDB is not ready yet. Run the DuckDB cells first.")


,minimum_membership,maximum_membership
0,0.0,0.999375


Sanity check passed: all fuzzy membership values are between 0 and 1.


## Troubleshooting

### 1. `FileNotFoundError` for CSV files

Check that the selected CSV files are inside a folder named `data` next to this notebook.

Expected structure:

```text
Tutorial 6.1/
├── Tutorial 6.1 - Sensor Dataset Preparation and Cleaning.ipynb
└── data/
    ├── 2026-04-26_22-34-20-119626_HR.csv
    ├── 2026-04-26_22-34-20-119626_T1.csv
    ├── 2026-04-26_22-34-20-119626_EA.csv
    ├── 2026-04-26_22-34-20-119626_SF.csv
    ├── 2026-04-26_22-34-20-119626_AX.csv
    ├── 2026-04-26_22-34-20-119626_AY.csv
    └── 2026-04-26_22-34-20-119626_AZ.csv
```

### 2. `ModuleNotFoundError`

Install the required Python packages inside your virtual environment:

```bash
pip install pandas duckdb matplotlib jupyterlab nbconvert
```

Make sure the package installation happens in the same environment where JupyterLab is running.

### 3. DuckDB database file issue

This notebook creates a local database file here:

```text
tutorial_data/database/tutorial_6_2_fuzzy.duckdb
```

If you want to restart from a clean database, close the notebook kernel and delete that file. It will be recreated when you run the DuckDB loading cell again.

### 4. Empty fuzzy query result

This does not always mean an error. It can mean that the threshold is too strict for this dataset. Try lowering the membership filter, for example from `> 0.60` to `> 0.30`.

### 5. Interpretation looks too simple

This is expected for the first version. The goal is to show how fuzzy linguistic variables can be created and queried. The interpretation rules can be refined after discussion.


## Summary

In this tutorial, you moved from a cleaned EmotiBit sensor table to a local DuckDB database and then to fuzzy linguistic interpretations.

You created:

- a DuckDB raw sensor table;
- a triangular fuzzy membership function as a SQL macro;
- five overlapping linguistic terms for heart rate, skin temperature, movement and arousal-related activity;
- fuzzy SQL queries;
- a combined sensor interpretation with plain-language explanations;
- a session-level linguistic summary;
- an automatic connection to the linguistic self-report saved in Tutorial 5.2;
- a human-readable comparison between sensor-derived words and human self-perception.

The important conceptual result is:

> Numerical sensor measurements remain precise, while fuzzy linguistic variables add gradual, human-readable meaning. This makes the sensor data easier to compare with the words used in a human self-report without claiming that the two must always agree.


## References

- Donald J. Norris, *Beginning Artificial Intelligence with the Raspberry Pi*, especially the fuzzy logic system chapter.
- Daniel Fasel, *Fuzzy Data Warehousing for Performance Measurement: Concept and Implementation*.
- Michael Kaufmann and Andreas Meier, *SQL and NoSQL Databases*, fuzzy databases section.
- EmotiBit exported CSV sensor files used in this tutorial.
- DuckDB documentation for SQL, views, and macros.
